# Stitch viewer: all tiles, design and contours, before and after stitching

Runs the pipeline on the chosen tiles and opens one interactive window (pyqtgraph, OpenGL):

1. **Per tile** (`process_tile`): load the SEM image, detect contacts (Otsu), refine the contours
   to the maximum gradient, read the design contacts from the tile's `.oas`.
2. **Stitching** (`stitch_tiles`): match the contacts in every overlap, robust B − A translation
   per pair, one translation correction per tile.
3. **Viewer** (`StitchViewer`): SEM images, design contours/centres, Otsu and refined
   contours/centres, overlap boxes and overlap outliers. *Placement* switches between the nominal
   (metadata) and the stitched positions; flip between them to see the correction. Layers and single
   tiles can be hidden on the right.

The window blocks this notebook until it is closed. Positions in µm.

In [ ]:
DATA_DIR = r"A:\Stitch_monitor\In_cell_without_PTM_20260914\260910_1143_Norm"
TILES = None              # None = all tiles, or a list of indices in read_tile_index order, e.g. list(range(10))
LAYER = (100, 0)          # (layer, datatype) of the contacts in the .oas
TONE_REVERSED = False     # True if the .oas files draw the area AROUND the holes
SEARCH_PX = 5.0           # refinement searches this many pixels in/out along the contour normal
STITCH_REFINED = True     # stitch with the refined centres (False: Otsu centres)
MAX_MATCH_NM = 10.0       # pair overlap contacts closer than this
OUTLIER_FACTOR = 3.0      # overlap outlier if residual > this x the median residual
MIN_MATCHED = 5           # skip pairs with fewer matched contacts (e.g. corner-only overlaps)
USE_OPENGL = True         # set False if the viewer window stays black

In [ ]:
import time

import numpy as np
import pandas as pd
import pyqtgraph as pg

from affine_ransac.io.design import load_layout, read_polygons
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.pipeline import process_tile, stitch_tiles
from affine_ransac.view_stitch import StitchViewer

## 1. Load the tiles and detect their contacts

In [ ]:
all_tiles = read_tile_index(DATA_DIR)
records = all_tiles if TILES is None else [all_tiles[i] for i in TILES]
tile_ids = [r.tile_id for r in records]

start = time.perf_counter()
tiles = [
    process_tile(r.image_path, r.oas_path, (r.center_x_nm, r.center_y_nm), (r.fov_x_nm, r.fov_y_nm),
                 LAYER, TONE_REVERSED, SEARCH_PX)
    for r in records
]
print(f"{len(tiles)} of {len(all_tiles)} tiles in {time.perf_counter() - start:.1f} s")
pd.DataFrame({
    "tile": tile_ids,
    "design": [len(t.design_centers) for t in tiles],
    "otsu": [len(t.otsu.centers) for t in tiles],
    "refined": [len(t.refined.centers) for t in tiles],
}).describe().round(1)

## 2. Stitching

In [ ]:
centers = np.array([t.center_nm for t in tiles])
fovs = np.array([t.fov_nm for t in tiles])
stitch = stitch_tiles([t.sem_points_nm(STITCH_REFINED) for t in tiles], centers, fovs,
                      MAX_MATCH_NM, OUTLIER_FACTOR, MIN_MATCHED)

print(f"{len(stitch.pairs)} overlapping pairs used")
print(f"Corrections: |x| max {np.abs(stitch.corrections[:, 0]).max():.2f} nm, "
      f"|y| max {np.abs(stitch.corrections[:, 1]).max():.2f} nm")
pd.DataFrame([{
    "tile_a": tile_ids[p.i], "tile_b": tile_ids[p.j], "matched": len(p.ia),
    "outliers": int((~p.fit.inliers).sum()), "shift_x_nm": p.fit.shift[0], "shift_y_nm": p.fit.shift[1],
    "rms_nm": p.fit.rms(),
} for p in stitch.pairs]).round(2)

## 3. Viewer (closes with the window)

In [ ]:
design_polygons = [read_polygons(load_layout(r.oas_path), *LAYER) if r.oas_path else [] for r in records]

app = pg.mkQApp("Stitch viewer")
viewer = StitchViewer(tiles, tile_ids, design_polygons, stitch, refined=STITCH_REFINED, use_opengl=USE_OPENGL)
viewer.show()
app.exec()